In [ ]:
# Linalg libraries
import numpy as np

# Plotting libraries
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm

# Data management libraries
import h5py as hf
from dataclasses import dataclass
import glob

In [ ]:
@dataclass
class ModelTrajectory:
    # Model data
    width: int
    depth: int
    activation: str
    lr: float
    architecture: str
    ds_size: float

    # Loss data
    train_loss: np.ndarray
    train_loss_std: np.ndarray
    test_loss: np.ndarray
    test_loss_std: np.ndarray

    # Accuracy data
    train_acc: np.ndarray
    train_acc_std: np.ndarray
    test_acc: np.ndarray
    test_acc_std: np.ndarray

    # CV data
    entropy: np.ndarray
    entropy_std: np.ndarray
    trace: np.ndarray
    trace_std: np.ndarray

In [ ]:
def load_model_trajectory(file_root, file_signature, accuracy = False):
    # Glob all files with the correct signature
    files = glob.glob(file_root + '/*' + file_signature + '*.h5')

    # Extract model parameters
    model_params = files[0].split("/")[-1].split('_')
    width = int(model_params[2])
    depth = int(model_params[3])
    activation = model_params[4]
    ds_size = float(model_params[5])    
    lr = float(model_params[7])
    architecture = model_params[1]

    # Load data
    train_losses = []
    test_losses = []

    if accuracy:
        train_accuracies = []
        test_accuracies = []
    
    entropies = []
    traces = []

    for file in files:
        with hf.File(file, 'r') as f:
            if file.split("/")[-1].split("_")[0] == "train":
                train_losses.append(f['loss'][:])
                if accuracy:
                    train_accuracies.append(f['accuracy'][:])
                entropies.append(f['entropy'][:])
                traces.append(f['trace'][:])
            else:
                test_losses.append(f['loss'][:])
                if accuracy:
                    test_accuracies.append(f['accuracy'][:])

    # Take the means, std and build the ModelTrajectory object
    train_loss = np.nanmean(train_losses, axis=0)
    test_loss = np.nanmean(test_losses, axis=0)
    train_loss_std = np.nanstd(train_losses, axis=0)
    test_loss_std = np.nanstd(test_losses, axis=0)

    if accuracy:
        train_accuracy = np.mean(train_accuracies, axis=0)
        test_accuracy = np.mean(test_accuracies, axis=0)

        train_accuracy_std = np.std(train_accuracies, axis=0)
        test_accuracy_std = np.std(test_accuracies, axis=0)

    entropy = np.nanmean(entropies, axis=0)
    trace = np.nanmean(traces, axis=0)
    entropy_std = np.std(entropies, axis=0)
    trace_std = np.std(traces, axis=0)

    return ModelTrajectory(
        width=width,
        depth=depth,
        activation=activation,
        lr=lr,
        architecture=architecture,
        ds_size=ds_size,
        train_loss=train_loss,
        train_loss_std=train_loss_std,
        test_loss=test_loss,
        test_loss_std=test_loss_std,
        entropy=entropy,
        entropy_std=entropy_std,
        trace=trace,
        trace_std=trace_std,
        train_acc=train_accuracy,
        train_acc_std=train_accuracy_std,
        test_acc=test_accuracy,
        test_acc_std=test_accuracy_std
    )

def load_model_trajectories(root_path):
    files = glob.glob(root_path + '/*.h5')

    # Get unique names
    unique_names = []
    for file in files:
        unique_names.append(
            "_".join(file.split('/')[-1].split('_')[2:7])
            )

    unique = np.unique(unique_names)

    # Load data
    data = []
    for name in unique:
        data.append(load_model_trajectory(root_path, name, accuracy=True))

    return data   

In [ ]:
# fuel_mse_order_two = load_model_trajectories("./mse-2")
# fuel_mse_order_four = load_model_trajectories("./mse-4")
data = load_model_trajectories("./adam-slow")

In [ ]:
fig, ax = plt.subplots(4, 3, figsize=(25, 25))

depths = [1, 2, 3]
widths = [4, 12, 32, 64]
activations = ["relu", "tanh", "sigmoid"]
line_styles = ['-', '--', '-.']
colours = ['b', 'r', 'g', 'y']

for row, width in enumerate(widths):
    # loss, trace, entropy
    # Column 1
    for depth in depths:
        for i, model in enumerate(data):
            if model.width == width and model.depth == depth:
                ax[row, 0].plot(model.train_loss, label=f"Depth {depth}", linestyle=line_styles[activations.index(model.activation)], color=colours[depths.index(depth)])
                # ax[row, 0].fill_between(
                #     np.arange(len(model.train_loss)),
                #     model.train_loss - model.train_loss_std,
                #     model.train_loss + model.train_loss_std,
                #     alpha=0.3
                # )
                ax[row, 0].set_yscale("log")

    for depth in depths:
        for i, model in enumerate(data):
            if model.width == width and model.depth == depth:
                ax[row, 1].plot(model.trace, label=f"Depth {depth}", linestyle=line_styles[activations.index(model.activation)], color=colours[depths.index(depth)])
                # ax[row, 1].fill_between(
                #     np.arange(len(model.trace)),
                #     model.train_loss - model.trace_std,
                #     model.train_loss + model.trace_std,
                #     alpha=0.3
                # )
                ax[row, 1].set_yscale("log")


    for depth in depths:
        for i, model in enumerate(data):
            if model.width == width and model.depth == depth:
                ax[row, 2].plot(model.entropy, label=f"Depth {depth}", linestyle=line_styles[activations.index(model.activation)], color=colours[depths.index(depth)])
                # ax[row, 2].fill_between(
                #     np.arange(len(model.entropy)),
                #     model.train_loss - model.entropy_std,
                #     model.train_loss + model.entropy_std,
                #     alpha=0.3
                # )


# Add title to first row of train loss, trace, entropy
ax[0, 0].set_title("Train Accuracy")
ax[0, 1].set_title("Trace")
ax[0, 2].set_title("Entropy")

# Add x label to bottom row of epoch
ax[-1, 0].set_xlabel("Epoch")
ax[-1, 1].set_xlabel("Epoch")
ax[-1, 2].set_xlabel("Epoch")

# Add network width as y label to all rows
for i, width in enumerate(widths):
    ax[i, 0].set_ylabel(f"Width: {width}")
    # ax[i, 1].set_ylabel(f"Width: {width}")
    # ax[i, 2].set_ylabel(f"Width: {width}")

# # Add y limit of 5 to all rows in first column
# for i in range(6):
#     ax[i, 0].set_ylim(None, 5)

# # Add y limit of 15000 to all rows of second column
# for i in range(6):
#     ax[i, 1].set_ylim(None, 200000)

# Make one custom legend for line colours and depths and one legend for line style and activations
red_patch = mpatches.Patch(color='red', label='1')
blue_patch = mpatches.Patch(color='blue', label='2')
green_patch = mpatches.Patch(color='green', label='3')

solid_patch = mpatches.Patch(color="k", linestyle="-", label="relu")
dashed_patch = mpatches.Patch(color="k", linestyle="--", label="tanh")
dot_dashed_patch = mpatches.Patch(color="k", linestyle="-.", label="sigmoid")

ax[0, 1].legend(handles=[red_patch, blue_patch, green_patch], title="Depth")
ax[0, 0].legend(handles=[solid_patch, dashed_patch, dot_dashed_patch], title="Activation")

# plt.savefig("conv-ave-pool.pdf")
plt.show()


In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(25, 10))

depths = [2, 3] #, 2, 3]
widths = [12, 32, 64] #, 12, 32, 64]
activations = ["relu"] #, "tanh", "sigmoid"]
line_styles = ['-', '--']
colours = ['b', 'r', 'g', 'y']

for row, width in enumerate(widths):
    for depth in depths:
        for i, model in enumerate(data):
            if model.width == width and model.depth == depth:
                ax[0].scatter(np.log(model.train_loss), model.entropy,  s=1, c=model.trace, norm=LogNorm())
                ax[0].scatter(np.log(model.test_loss), model.entropy,  s=1, c="k", norm=LogNorm())

                ax[1].scatter(np.log(model.train_loss), np.log(model.trace), s=1, c=model.entropy)
                ax[1].scatter(np.log(model.test_loss), np.log(model.trace),  s=1, c="k", norm=LogNorm())


                ax[2].plot(model.train_loss, '.')
                ax[2].plot(model.test_loss)



ax[0].set_ylabel("log(S)")
ax[0].set_xlabel("log(Train Loss)")
ax[0].set_xlim(2.0, -20)
# ax[0].set_ylim(-2, 2)
plt.colorbar(ax[0].collections[0], ax=ax[0], label="Trace")

ax[1].set_xlabel("log(Train Loss)")
ax[1].set_ylabel("log(Trace)")
ax[1].set_xlim(2.0, -20)

plt.colorbar(ax[1].collections[0], ax=ax[1], label="Entropy")

ax[2].set_xlabel("log(Trace)")
# ax[2].set_ylim(-3, 2)
ax[2].set_ylabel("log(S)")
# plt.colorbar(ax[2].collections[0], ax=ax[2], label="Train Loss")

# plt.savefig("mnist-phases.pdf")
plt.show()